## Clean pandas

In [1]:
import pandas as pd
import numpy as np
from time import time

In [2]:
N = 100_000_000
G = 1_000  # число групп

df = pd.DataFrame({
    "id": np.random.randint(0, G, size=N),
    "value": np.random.randn(N).astype("float64"),
})

In [3]:
def slow_func(x: pd.Series) -> pd.Series:
    # что-то не векторизованное, чистый Python
    m = x.mean()
    s = x.std()
    return (x - m) / s

In [4]:
%%time
agg = df.groupby("id")["value"].agg(["sum", "mean", "std"])
norm = df.groupby("id")["value"].transform(slow_func)

CPU times: user 11.7 s, sys: 5.69 s, total: 17.4 s
Wall time: 20 s


## Polars

Polars — колоночный движок на Rust, многопоточный «из коробки». Практически всегда быстрее pandas на тяжёлых groupby/agg/joins.

Типичный выигрыш:
для 10–100 млн строк Polars может быть в 3–10 раз быстрее, плюс лучше масштабируется по ядрам.

In [5]:
import polars as pl

In [6]:
df_pl = pl.DataFrame({
    "id": np.random.randint(0, G, size=N),
    "value": np.random.randn(N),
})

In [7]:
%%time
agg = (
    df_pl
    .group_by("id")
    .agg([
        pl.col("value").sum().alias("sum"),
        pl.col("value").mean().alias("mean"),
        pl.col("value").std().alias("std"),
    ])
)

# Нормализация внутри группы — полностью векторизованно:
norm = (
    df_pl
    .with_columns([
        (
        (pl.col("value") - pl.col("value").mean().over("id")) /
        pl.col("value").std().over("id")
        ).alias("value_norm")
    ])
)

CPU times: user 14.5 s, sys: 3.22 s, total: 17.7 s
Wall time: 3.34 s
